# PSMA pKi Activity Classification — Full Enamine Library Screen (F10)

A deliberately small pipeline with one job: train the two models that survived F9's
evaluation, then put an **active / inactive** call on every compound in the 290,000-member
Enamine library.

## What this is, and what it is not

F3–F9 were about *measuring* whether the modelling works. That question is now answered,
so this notebook does not re-ask it. It keeps only the parts needed to produce a defensible
prediction, and drops everything else: the applicability domain, the enumeration, the
multi-task network, the ChEMBL pretraining, five of the seven classifiers, the equal-weight
blend, the atom attribution and the descriptor machinery.

**The applicability domain is deliberately gone.** F6–F9 established that essentially none
of this catalogue is within structural reach of the PSMA training set — the AD put 36 of
~200,000 compounds in range in F7, and 1.37% of a *deliberately near-sampled* subset in F9.
That is a real chemistry mismatch, not a fixable modelling defect: PSMA ligands are polar
Glu-urea peptidomimetics and a lead-like diversity deck contains very little resembling
them. Running the AD again would only reprint that conclusion and veto the entire output.
So every prediction here is an **extrapolation**, and the ranking is the deliverable rather
than the probability. This is stated once, here, and the numbers are reported straight
afterwards.

## The two models, and why these two

F9's leave-one-cluster-out evaluation, pooled over 278 compounds with whole chemical
families held out:

| model | ROC-AUC | PR-AUC | precision@50 | MCC | ECE (raw) |
|---|---|---|---|---|---|
| **LightGBM** | **0.804** | 0.762 | 0.80 | 0.454 | 0.114 |
| **RandomForest** | 0.777 | 0.716 | 0.74 | **0.478** | **0.043** |
| KPLS-DA | 0.775 | 0.766 | 0.82 | 0.375 | 0.193 |
| SVM (Tanimoto) | 0.763 | 0.730 | 0.76 | 0.441 | 0.059 |
| LogReg (L2) | 0.763 | 0.735 | 0.80 | 0.417 | 0.052 |
| kNN (Tanimoto) | 0.712 | 0.683 | 0.76 | 0.301 | 0.168 |
| BernoulliNB | 0.686 | 0.654 | 0.72 | 0.187 | 0.389 |

**LightGBM** was the best discriminator and had the smallest optimism gap between a random
split and a grouped one (−0.061, against −0.104 for logistic regression and −0.129 for
nearest-neighbour lookup). **RandomForest** was second on AUC, best on MCC, and the only
strong model whose probabilities needed no recalibration at all (ECE 0.043). They are also
a genuinely different pair — boosting against bagging — so their disagreement carries
information, which is the basis of the three-tier call below.

KPLS-DA was the close third and was actually the best model at the very top of the ranking
(precision@50 0.82, PR-AUC 0.766). It was left out because its probabilities were the worst
calibrated of the strong models (ECE 0.193) and its precision@50 edge over LightGBM is one
compound in fifty — well inside noise at n = 278. If you want a third opinion, it is the one
to add.

**The blend is gone.** Equal-weight blending failed to beat its best single member on seven
consecutive endpoints across F4–F9 (here, 0.783 against LightGBM's 0.804). The two models
are reported separately and combined only as a consensus *call*, not as an averaged score.

## Fingerprints only

F9's feature-set bake-off on pKi: `fp_only` 0.865, `fp_plus_descriptors` 0.850,
`descriptors_only` 0.844. The 70 curated RDKit descriptors were adding nothing — the same
result as in F6 and F7 — so they are dropped. That is not only a simplification: computing
217 descriptors per molecule (BCUT2D needs an eigendecomposition each) over 290,000
compounds is by far the most expensive step in a screen this size, and it buys negative
accuracy. Morgan radius 2, 2048 bits, and nothing else.

## Where the active/inactive line comes from

A classifier outputs a probability; calling something "active" needs a threshold, and 0.5 is
an arbitrary choice that happens to be near-optimal only when the base rate is 0.5 and the
costs are symmetric. Here the threshold is **chosen from out-of-fold predictions** — the
same leave-one-cluster-out predictions used to measure the models — by maximising MCC. Part
D prints the full threshold sweep, so you can instead pick the threshold that hits a
precision target, and reports what each choice implies for how many of the 290,000 get
called. The calibrator is fitted on those same out-of-fold predictions and applied once at
deployment, so the probability column means something and the threshold transfers.

Deriving the threshold from the training fit instead would put it wherever the models
overfit to, which on this data is around 0.99 train AUC.

## Running order

- **Part A** — BindingDB pKi set, with the Cheng–Prusoff-derived values removed and the
  censored measurements recovered (unchanged from F9, and load-bearing: 141 of 169
  dual-measured compounds carried arithmetic pKi values).
- **Part B** — Morgan fingerprints, InChIKey deduplication.
- **Part C** — the two models, lightly re-tuned under `fp_only`.
- **Part D** — leave-one-cluster-out evaluation, the threshold sweep, calibration.
- **Part E** — stream the 290,000-compound SDF and score it in chunks.
- **Part F** — the calls, model agreement, and the shortlist.

Expect Part E to be the long one: roughly 15–40 minutes for fingerprinting, depending on the
machine. Everything before it runs in a couple of minutes.

## 0. Setup

In [ ]:
%pip install -q duckdb rdkit pandas numpy scikit-learn matplotlib seaborn requests tqdm scipy joblib lightgbm

In [ ]:
import os
import io
import re
import gzip
import json
import zipfile
import shutil
import warnings
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional, Sequence, List, Dict, Tuple

import duckdb
import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit import DataStructs

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             balanced_accuracy_score, matthews_corrcoef,
                             brier_score_loss, precision_score, recall_score, f1_score)
import joblib

from lightgbm import LGBMClassifier

import matplotlib.pyplot as plt
import seaborn as sns

RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
np.random.seed(0)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

print("Environment ready. Two models, fingerprints only, no applicability domain.")

## 1. Configuration

Everything that changes behaviour is here. The fields most likely to want changing are
`screening_library_path`, `screening_limit` (set it to a few thousand for a trial run),
`threshold_objective` / `threshold_override`, and `retune`.

In [ ]:
@dataclass
class Config:
    # ---- Target -------------------------------------------------------------------
    uniprot_id: str = "Q04609"               # FOLH1 / PSMA / glutamate carboxypeptidase II
    target_name: str = "PSMA"

    # ---- Endpoints ----------------------------------------------------------------
    # Both are PULLED, only pKi is MODELLED. pIC50 is needed by the derived-value
    # detector in Part A.2: 141 of 169 dual-measured compounds carry a pKi that was
    # computed from the IC50 by Cheng-Prusoff rather than measured, and dropping those
    # is the single most consequential piece of data cleaning in this pipeline.
    endpoint_names: Sequence[str] = field(default_factory=lambda: ["pIC50", "pKi"])
    affinity_columns: Sequence[str] = field(
        default_factory=lambda: ["IC50 (nM)", "Ki (nM)"])
    primary_endpoint: str = "pKi"
    derived_endpoint: str = "pKi"             # the endpoint whose values may be computed
    active_threshold: float = 8.0             # pKi >= 8.0, i.e. Ki <= 10 nM
    min_affinity_nm: float = 1e-3
    max_affinity_nm: float = 1e6
    keep_censored_as_labels: bool = True
    drop_derived_endpoint_values: bool = True
    derived_min_block_size: int = 5
    derived_offset_tol: float = 0.03

    # ---- Fingerprints -------------------------------------------------------------
    # fp_only. F9's bake-off: fp_only 0.865, fp_plus_descriptors 0.850,
    # descriptors_only 0.844 -- the descriptors were adding nothing, and at 290K
    # compounds they would dominate the runtime.
    morgan_radius: int = 2
    morgan_n_bits: int = 2048
    dedupe_on_inchikey: bool = True

    # ---- Models -------------------------------------------------------------------
    models: Sequence[str] = field(default_factory=lambda: ["LightGBM", "RandomForest"])
    class_weight_balanced: bool = True
    retune: bool = True                       # False reuses the F9 hyperparameters below
    tuning_cv_folds: int = 10
    n_bootstrap_shortlist: int = 20            # bootstrap spread, top hits only

    # ---- Evaluation and the decision threshold ------------------------------------
    butina_cutoff: float = 0.4
    loco_min_cluster_size: int = 25
    loco_min_fold_actives: int = 5
    loco_max_clusters: int = 12
    loco_n_remainder_folds: int = 3
    threshold_objective: str = "mcc"          # "mcc" | "f1" | "balanced_acc"
    threshold_override: Optional[float] = None  # set a number to force the cut
    # A cut is only allowed if it calls at most this fraction active. Without it, MCC
    # and F1 are both happy to maximise at a very permissive threshold when the model is
    # weak -- the smoke run chose p >= 0.11, calling 86% of compounds active at recall
    # 1.000, which over 290,000 compounds is a 250,000-compound "shortlist". The point of
    # a screen is to decide what NOT to look at.
    threshold_max_called_frac: float = 0.25
    calibrate: bool = True                    # Platt, fitted on out-of-fold predictions
    precision_at_n: Sequence[int] = field(default_factory=lambda: [50, 100, 200])
    enrichment_fractions: Sequence[float] = field(
        default_factory=lambda: [0.01, 0.05, 0.10])

    # ---- The library --------------------------------------------------------------
    screening_library_path: str = "/Enamine_screen_library.sdf"
    screening_chunk_size: int = 25000         # molecules held in memory at once
    screening_limit: Optional[int] = None     # set e.g. 20000 for a quick trial run
    shortlist_size: int = 500

    # ---- Paths --------------------------------------------------------------------
    data_dir: str = "./ad_pipeline_data"
    output_dir: str = "./ad_pipeline_outputs/f10"
    bindingdb_tsv_url: str = (
        "https://bindingdb.org/bind/downloads/BindingDB_All_202409_tsv.zip")
    random_state: int = 42

    def __post_init__(self):
        Path(self.data_dir).mkdir(parents=True, exist_ok=True)
        Path(self.output_dir).mkdir(parents=True, exist_ok=True)


CONFIG = Config()
PRIMARY = CONFIG.primary_endpoint
POSITIVE_CLASS = 1
CONFIG

## 2. Part A.1 — BindingDB PSMA set

Resolves and caches the BindingDB release, queries it with DuckDB for every measurement
against UniProt Q04609 (FOLH1 / PSMA / glutamate carboxypeptidase II), and pivots to a
wide per-compound table.

Both IC50 and Ki are pulled. Only Ki is modelled — the IC50 column exists so that Part A.2
can detect which Ki values were computed from it rather than measured, which turns out to
apply to 83% of the compounds measured both ways.

In [ ]:
def resolve_bindingdb_tsv_url(cfg: Config) -> tuple:
    '''Scrape the BindingDB download page for the CURRENT 'BindingDB_All_<YYYYMM>_tsv.zip'
    filename (BindingDB republishes this monthly, so a hardcoded name goes stale).'''
    download_page = "https://www.bindingdb.org/rwd/bind/chemsearch/marvin/Download.jsp"
    try:
        resp = requests.get(download_page, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
        resp.raise_for_status()
        m = re.search(r"download_file=(/rwd/bind/downloads/BindingDB_All_\d{6}_tsv\.zip)", resp.text)
        if m:
            file_param = m.group(1)
            zip_name = file_param.rsplit("/", 1)[-1]
            direct_url = f"https://www.bindingdb.org{file_param}"
            print(f"Resolved current BindingDB TSV: {zip_name}")
            return direct_url, zip_name
        print(f"Could not find the current filename on {download_page}; using CONFIG default.")
    except Exception as e:
        print(f"Could not reach {download_page} ({e}); using CONFIG default.")
    return cfg.bindingdb_download_url, cfg.bindingdb_tsv_zip_name


def _fetch_to_file(url: str, dest: Path) -> str:
    headers = {"User-Agent": "Mozilla/5.0 (compatible; AD-pipeline/1.0)"}
    with requests.get(url, stream=True, timeout=120, headers=headers) as r:
        r.raise_for_status()
        content_type = r.headers.get("content-type", "")
        total = int(r.headers.get("content-length", 0))
        with open(dest, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=dest.name) as pbar:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
                pbar.update(len(chunk))
    return content_type


def _recover_real_download_link(html_path: Path) -> Optional[str]:
    try:
        text = html_path.read_text(errors="ignore")
    except Exception:
        return None
    m = re.search(r'(https://www\.bindingdb\.org/rwd/bind/downloads/[^\s"\x27<>)]+\.zip)', text)
    return m.group(1) if m else None


def download_bindingdb(cfg: Config) -> Optional[Path]:
    '''Download (if needed) and unzip the full BindingDB TSV dump. Returns None (instead of
    raising) if the download can't be completed, so the notebook can fall back to a
    synthetic fine-tuning set and still run end-to-end.'''
    data_dir = Path(cfg.data_dir)
    try:
        if cfg.auto_resolve_bindingdb_url:
            url, zip_name = resolve_bindingdb_tsv_url(cfg)
        else:
            url, zip_name = cfg.bindingdb_download_url, cfg.bindingdb_tsv_zip_name

        zip_path = data_dir / zip_name
        tsv_path = data_dir / zip_name.replace(".zip", "")

        if tsv_path.exists() and not cfg.force_redownload:
            print(f"Found existing TSV: {tsv_path}")
            return tsv_path

        if not zip_path.exists() or cfg.force_redownload:
            print(f"Downloading {url} ...")
            content_type = _fetch_to_file(url, zip_path)
            print(f"Saved to {zip_path} (content-type: {content_type})")

            if not zipfile.is_zipfile(zip_path):
                real_url = _recover_real_download_link(zip_path)
                if real_url and real_url != url:
                    print(f"Not a zip — retrying with the real link found on the landing page: {real_url}")
                    _fetch_to_file(real_url, zip_path)

        if not zipfile.is_zipfile(zip_path):
            with open(zip_path, "rb") as f:
                head = f.read(300)
            zip_path.unlink(missing_ok=True)
            raise RuntimeError(f"Downloaded file is not a valid zip. First bytes: {head!r}")

        print("Unzipping ...")
        with zipfile.ZipFile(zip_path) as zf:
            member = [m for m in zf.namelist() if m.lower().endswith(".tsv")][0]
            with zf.open(member) as src, open(tsv_path, "wb") as dst:
                shutil.copyfileobj(src, dst)
        print(f"Extracted: {tsv_path}")
        return tsv_path

    except Exception as e:
        print(f"Could not obtain the BindingDB TSV dump ({e}).")
        return None


def query_bindingdb_duckdb_multitask(tsv_path: Path, cfg: Config) -> pd.DataFrame:
    '''Pull one long dataframe with a row per (compound, endpoint) for every column in
    `cfg.affinity_columns`, using DuckDB SQL streaming over the TSV (never materialized
    fully in pandas).'''
    con = duckdb.connect(database=":memory:")
    con.execute("PRAGMA threads=4;")
    con.execute(f'''
        CREATE OR REPLACE VIEW bdb AS
        SELECT * FROM read_csv(
            '{tsv_path.as_posix()}', delim='\t', header=True, quote='"', escape='"',
            strict_mode=False, ignore_errors=True, all_varchar=True
        )
    ''')

    base_where = ['"Ligand SMILES" IS NOT NULL']
    if cfg.target_uniprot_id:
        uniprot_conditions = [
            f'"UniProt (SwissProt) Primary ID of Target Chain {i}" = \'{cfg.target_uniprot_id}\''
            for i in range(1, 6)
        ]
        base_where.append("(" + " OR ".join(uniprot_conditions) + ")")
    if cfg.target_name_like:
        safe = cfg.target_name_like.replace("'", "''")
        base_where.append(f'"Target Name" ILIKE \'%{safe}%\'')
    if cfg.organism_like:
        safe = cfg.organism_like.replace("'", "''")
        base_where.append(f'"Target Source Organism According to Curator or DataSource" ILIKE \'%{safe}%\'')

    frames = []
    for col, endpoint in zip(cfg.affinity_columns, cfg.endpoint_names):
        where_sql = " AND ".join(base_where + [f'"{col}" IS NOT NULL'])
        query = f'''
            SELECT
                "Ligand SMILES"                                    AS smiles,
                "BindingDB Ligand Name"                            AS ligand_name,
                "{col}"                                            AS affinity_raw,
                '{endpoint}'                                       AS endpoint_name
            FROM bdb
            WHERE {where_sql}
        '''
        part = con.execute(query).fetch_df()
        print(f"  {endpoint} ('{col}'): {len(part):,} raw rows.")
        frames.append(part)
    con.close()
    return pd.concat(frames, ignore_index=True)


def _synthetic_bindingdb_finetune_set(cfg: Config, n: int = 250) -> pd.DataFrame:
    print("!! Using SYNTHETIC BindingDB PSMA fine-tuning data (network/download unavailable) !!")
    demo_smiles = [
        "CCOc1ccc2nc(S(N)(=O)=O)sc2c1", "CC(C)Cc1ccc(C(C)C(=O)O)cc1",
        "O=C(O)c1ccccc1O", "CC(=O)Oc1ccccc1C(=O)O",
        "Clc1ccc(C(c2ccccc2Cl)N2CCN(CCO)CC2)cc1",
        "COc1cc2c(cc1OC)C(=O)C(CC1CCN(C)CC1)C2",
        "CN1CCC[C@H]1c1cccnc1", "CC(C)NCC(O)COc1cccc2ccccc12",
        "NC(CCC(=O)O)C(=O)O", "OC(=O)CCC(NC(=O)NC(CCC(=O)O)C(=O)O)C(=O)O",
    ]
    rng = np.random.RandomState(cfg.random_state)
    rows = []
    for i in range(n):
        smi = demo_smiles[i % len(demo_smiles)]
        row = {"smiles": smi, "ligand_name": f"synthetic_{i}"}
        if rng.rand() < 0.75:
            row[cfg.endpoint_names[0]] = float(np.clip(rng.normal(6.8, 1.2), 3, 10))
        if rng.rand() < 0.6:
            row[cfg.endpoint_names[1]] = float(np.clip(rng.normal(6.5, 1.3), 3, 10))
        rows.append(row)
    return pd.DataFrame(rows)


def clean_bindingdb_multitask(df_long: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    '''Coerce to numeric, drop censored (>,<) measurements, compute pAffinity, take the
    per-compound median for each endpoint separately, then pivot wide so every
    compound is one row with (possibly-NaN) pIC50 / pKi columns.'''
    df = df_long.copy()
    df["is_censored"] = df["affinity_raw"].astype(str).str.contains("[<>]", regex=True)
    df = df[~df["is_censored"]]
    df["affinity_nm"] = pd.to_numeric(df["affinity_raw"], errors="coerce")
    df = df.dropna(subset=["affinity_nm", "smiles"])
    df = df[(df["affinity_nm"] >= cfg.min_affinity_nm) & (df["affinity_nm"] <= cfg.max_affinity_nm)]
    df["pAffinity"] = 9 - np.log10(df["affinity_nm"])

    per_endpoint = (
        df.groupby(["smiles", "endpoint_name"], as_index=False)
        .agg({"pAffinity": "median", "ligand_name": "first"})
    )
    wide = per_endpoint.pivot_table(index="smiles", columns="endpoint_name", values="pAffinity")
    names = per_endpoint.groupby("smiles")["ligand_name"].first()
    wide = wide.join(names).reset_index()
    for ep in cfg.endpoint_names:
        if ep not in wide.columns:
            wide[ep] = np.nan
    print(f"Cleaned multi-task fine-tuning set: {len(wide):,} unique compounds "
          + ", ".join(f"({ep}: {wide[ep].notna().sum():,})" for ep in cfg.endpoint_names))
    return wide.reset_index(drop=True)


# --- run it -------------------------------------------------------------------
tsv_path = download_bindingdb(CONFIG)
if tsv_path is not None:
    raw_bdb_df = query_bindingdb_duckdb_multitask(tsv_path, CONFIG)
    train_df = clean_bindingdb_multitask(raw_bdb_df, CONFIG)
else:
    print("!" * 90)
    print("!! THE BINDINGDB DOWNLOAD FAILED — falling back to a SYNTHETIC dataset.")
    print("!! Everything below will run and produce plausible-looking numbers that are")
    print("!! about randomly generated molecules. Fix the download before reading any")
    print("!! result. CONFIG.bindingdb_tsv_url is the first thing to check.")
    print("!" * 90)
    train_df = _synthetic_bindingdb_finetune_set(CONFIG)
    # Part A.2 re-derives labels from the long frame, so it must exist in both branches.
    _long = []
    for _r in train_df.itertuples(index=False):
        for _ep, _col in zip(CONFIG.endpoint_names, CONFIG.endpoint_names):
            _v = getattr(_r, _col, np.nan)
            if pd.notna(_v):
                _long.append({"smiles": _r.smiles, "ligand_name": _r.ligand_name,
                              "affinity_raw": f"{10 ** (9 - float(_v)):.6g}",
                              "endpoint_name": _ep})
    raw_bdb_df = pd.DataFrame(_long)
    IS_SYNTHETIC = True

IS_SYNTHETIC = bool(globals().get("IS_SYNTHETIC", False))
assert "raw_bdb_df" in globals(), "raw_bdb_df was not built — Part A.2 cannot run"
print(f"\nraw_bdb_df: {raw_bdb_df.shape}   wide train_df: {train_df.shape}"
      + ("   [SYNTHETIC]" if IS_SYNTHETIC else ""))
train_df.head()

## 2b. Part A.2 — pKi labels

Unchanged from F9 in substance, trimmed to one endpoint in output. Four things happen here,
and the first two are the ones that matter:

1. **Derived-value detection.** Blocks of compounds whose pIC50 and pKi differ by a *fixed*
   amount were not measured twice — the second value was computed from the first by the
   Cheng–Prusoff equation. In this dataset that is 141 of 169 dual-measured compounds (83%),
   in three blocks: 106 at a ratio of 140.1 from US20250213705, 30 at 5.01 from US10736974,
   and 5 more at 130.4 from the first filing again. Those pKi values carry no information
   beyond the IC50 they came from, and they would put a large block of the pKi column on a
   different scale from the rest, making any single threshold wrong for most of it. They are
   dropped; the affected compounds keep their pIC50, which is simply not modelled here.

2. **Censored-measurement recovery.** A `>` bound at or below the active cut proves the
   compound is not active, and a `<` bound at or above it proves that it is. Both become
   usable labels instead of being discarded. Bounds that straddle the cut are genuinely
   uninformative and are dropped.

3. **Provenance tagging** from the BindingDB ligand name, so Part D can hold a single patent
   filing out as its own fold rather than letting it leak across the evaluation.

4. **Median aggregation** over replicate measurements of the same structure.

Compounds with a pIC50 but no usable pKi are dropped here, not carried. F9 kept them to
widen the applicability domain and shape the clustering; with the AD gone, the only thing
they would still affect is the Butina clustering in Part D, and letting unlabelled
compounds define the fold boundaries makes the folds harder to interpret for no gain.

In [ ]:
# --- pKi label assembly ----------------------------------------------------------------
EPS = [PRIMARY]
EP_COLS = dict(zip(CONFIG.endpoint_names, CONFIG.affinity_columns))
A_CUT = float(CONFIG.active_threshold)

GENERIC_SOURCE_TAGS = {"CHEMBL", "other", "unknown"}
_PATENT_RE = re.compile(r"\b((?:US|WO|EP|CN|JP|KR)\s?\d{6,})", re.I)
_CHEMBL_RE = re.compile(r"\b(CHEMBL\d+)", re.I)


def source_tag(name) -> str:
    """Coarse provenance from the ligand name. A patent number identifies one filing;
    CHEMBL and 'other' are catch-all buckets and must not be read as a single source."""
    if not isinstance(name, str) or not name.strip():
        return "unknown"
    m = _PATENT_RE.search(name)
    if m:
        return m.group(1).replace(" ", "").upper()
    if _CHEMBL_RE.search(name):
        return "CHEMBL"
    return "other"


print(f"Modelling {PRIMARY} only.  active >= {A_CUT:.2f}  "
      f"({10 ** (9 - A_CUT):.3g} nM or better)")

# =====================================================================================
# 1. Derived-value detection
# =====================================================================================
def detect_derived_endpoint(df_wide, ep_a, ep_b, min_block=5, tol=0.03):
    """Blocks of compounds whose two endpoints differ by a FIXED amount.

    Candidate offsets whose means fall within `tol` of an already-accepted block are
    merged into it, so one block is not reported three times under adjacent rounding
    buckets. Returns (report, mask)."""
    empty = (pd.DataFrame(), pd.Series(False, index=df_wide.index))
    if ep_a not in df_wide.columns or ep_b not in df_wide.columns:
        return empty
    both = df_wide[df_wide[ep_a].notna() & df_wide[ep_b].notna()].copy()
    if len(both) < min_block:
        return empty
    both["_diff"] = both[ep_b] - both[ep_a]

    blocks, claimed = [], set()
    for val, _n in both["_diff"].round(2).value_counts().items():
        sel = both[((both["_diff"] - val).abs() <= tol) & (~both.index.isin(claimed))]
        if len(sel) < min_block:
            continue
        if any(abs(b["offset"] - float(sel["_diff"].mean())) <= tol * 1.5 for b in blocks):
            continue
        srcs = (sel["source_tag"].value_counts() if "source_tag" in sel
                else pd.Series(dtype=int))
        blocks.append({
            "offset": round(float(sel["_diff"].mean()), 4),
            "implied_ratio": round(float(10 ** sel["_diff"].mean()), 2),
            "n_compounds": len(sel),
            "offset_sd": round(float(sel["_diff"].std(ddof=0)), 5),
            "dominant_source": srcs.index[0] if len(srcs) else "?",
            "source_share": round(float(srcs.iloc[0] / len(sel)), 3) if len(srcs) else np.nan,
        })
        claimed.update(sel.index)

    mask = pd.Series(False, index=df_wide.index)
    mask.loc[sorted(claimed)] = True
    report = (pd.DataFrame(blocks).sort_values("n_compounds", ascending=False)
              if blocks else pd.DataFrame())
    return report, mask


_wide = train_df.copy()
_wide["source_tag"] = _wide["ligand_name"].map(source_tag)
derived_report, derived_mask = detect_derived_endpoint(
    _wide, CONFIG.endpoint_names[0], CONFIG.endpoint_names[1],
    min_block=CONFIG.derived_min_block_size, tol=CONFIG.derived_offset_tol)
derived_smiles = set(_wide.loc[derived_mask, "smiles"])

print("\n" + "=" * 90)
print("DERIVED-VALUE DETECTOR — fixed-offset blocks between pIC50 and pKi")
print("=" * 90)
if len(derived_report):
    display(derived_report)
    _n_pairs = int((_wide[CONFIG.endpoint_names[0]].notna()
                    & _wide[CONFIG.endpoint_names[1]].notna()).sum())
    print(f"{int(derived_mask.sum())} of {_n_pairs} dual-measured compounds "
          f"({100 * derived_mask.sum() / max(_n_pairs, 1):.0f}%) sit in a fixed-offset "
          f"block.")
    print("A constant offset across many compounds from one filing means the pKi was")
    print("computed from the IC50 (Cheng-Prusoff), not measured.")
    print("DROPPED." if CONFIG.drop_derived_endpoint_values else
          "KEPT (drop_derived_endpoint_values is False) — expect a bimodal pKi column.")
else:
    print("No fixed-offset blocks found. The endpoints look independently measured.")


# =====================================================================================
# 2. Label assembly from the raw pull, keeping the censored rows
# =====================================================================================
def build_labels(ep: str, a_cut: float):
    col = EP_COLS[ep]
    sub = raw_bdb_df[raw_bdb_df["endpoint_name"] == ep].copy()
    raw = sub["affinity_raw"].astype(str)
    sub["qualifier"] = raw.str.extract(r"^\s*([<>]=?)")[0]
    sub["value_nm"] = pd.to_numeric(raw.str.replace(r"^\s*[<>]=?\s*", "", regex=True),
                                    errors="coerce")
    sub = sub.dropna(subset=["value_nm", "smiles"])
    sub = sub[(sub["value_nm"] >= CONFIG.min_affinity_nm)
              & (sub["value_nm"] <= CONFIG.max_affinity_nm)]
    sub["pAff"] = 9 - np.log10(sub["value_nm"])

    numeric = sub[sub["qualifier"].isna()]
    gt = sub[sub["qualifier"].notna() & sub["qualifier"].str.startswith(">")]
    lt = sub[sub["qualifier"].notna() & sub["qualifier"].str.startswith("<")]

    num_agg = (numeric.groupby("smiles", as_index=False)
               .agg(pAff=("pAff", "median"), ligand_name=("ligand_name", "first"),
                    n_meas=("pAff", "size")))
    # several '>' bounds must all hold, so the most informative is the smallest
    gt_agg = (gt.groupby("smiles", as_index=False)
              .agg(p_upper=("pAff", "min"), ligand_name=("ligand_name", "first")))
    lt_agg = (lt.groupby("smiles", as_index=False)
              .agg(p_lower=("pAff", "max"), ligand_name=("ligand_name", "first")))

    rec, stats = {}, {"numeric_rows": len(numeric), "numeric_cpds": len(num_agg),
                      "gt_rows": len(gt), "lt_rows": len(lt), "recovered_inactive": 0,
                      "recovered_active": 0, "dropped_ambiguous": 0, "dropped_derived": 0}

    _drop_derived = (CONFIG.drop_derived_endpoint_values
                     and ep == CONFIG.derived_endpoint)

    for r in num_agg.itertuples(index=False):
        if _drop_derived and r.smiles in derived_smiles:
            stats["dropped_derived"] += 1
            continue
        rec[r.smiles] = {"ligand_name": r.ligand_name, "pAff": float(r.pAff),
                         "y": int(r.pAff >= a_cut), "label_source": "numeric",
                         "n_meas": int(r.n_meas)}

    if CONFIG.keep_censored_as_labels:
        for r in gt_agg.itertuples(index=False):
            if r.smiles in rec:
                continue
            if _drop_derived and r.smiles in derived_smiles:
                stats["dropped_derived"] += 1
                continue
            if r.p_upper <= a_cut:                       # provably not active
                rec[r.smiles] = {"ligand_name": r.ligand_name, "pAff": np.nan, "y": 0,
                                 "label_source": "censored", "n_meas": 0}
                stats["recovered_inactive"] += 1
            else:
                stats["dropped_ambiguous"] += 1
        for r in lt_agg.itertuples(index=False):
            if r.smiles in rec:
                continue
            if r.p_lower >= a_cut:                       # provably active
                rec[r.smiles] = {"ligand_name": r.ligand_name, "pAff": np.nan, "y": 1,
                                 "label_source": "censored", "n_meas": 0}
                stats["recovered_active"] += 1
            else:
                stats["dropped_ambiguous"] += 1
    return rec, stats


records, label_stats = build_labels(PRIMARY, A_CUT)
print("\n" + "=" * 90)
print("LABEL ASSEMBLY")
print("=" * 90)
display(pd.DataFrame({PRIMARY: label_stats}).T)

train_df = pd.DataFrame([
    {"smiles": smi, PRIMARY: r["pAff"], "y": int(r["y"]),
     "label_source": r["label_source"], "n_meas": r["n_meas"],
     "ligand_name": r["ligand_name"]}
    for smi, r in records.items()
]).reset_index(drop=True)
train_df["source_tag"] = train_df["ligand_name"].map(source_tag)

y_all = train_df["y"].values.astype(int)
print(f"\nModelled compounds : {len(train_df):,}")
print(f"  active   (pKi >= {A_CUT:.1f}) : {int((y_all == 1).sum()):,}")
print(f"  inactive                  : {int((y_all == 0).sum()):,}")
print(f"  base rate                 : {y_all.mean():.3f}")
print(f"  from censored bounds      : "
      f"{int((train_df['label_source'] == 'censored').sum()):,}")

print("\nSource composition:")
_src = train_df["source_tag"].value_counts()
for tag, n in _src.head(8).items():
    print(f"  {tag:>16s}: {n:4d} ({100 * n / len(train_df):5.1f}%)")
_specific = _src[~_src.index.isin(GENERIC_SOURCE_TAGS)]
if len(_specific) and _specific.iloc[0] / len(train_df) > 0.15:
    print(f"\n!! '{_specific.index[0]}' — a single filing — supplies {_specific.iloc[0]} "
          f"of {len(train_df)} compounds ({100 * _specific.iloc[0] / len(train_df):.0f}%).")
    print("   Part D holds it out as its own fold so its contribution is measurable.")

train_df.head()

## 3. Part B — Fingerprints

Morgan radius 2, 2048 bits, and nothing else. Standardisation (normalise → largest fragment
→ uncharge) runs first so that the training set and the library are described on the same
footing, and deduplication is on the standardised InChIKey — in F7 31% of this catalogue
turned out to be protonation and tautomer variants of structures already in it, which would
otherwise inflate the library count and the apparent hit count alike.

One deliberate asymmetry: the training set is standardised and deduplicated here, in memory,
while the library is standardised chunk by chunk in Part E and deduplicated on the way
through using a running key set. 290,000 InChIKeys is cheap to hold; 290,000 RDKit molecules
is not.

In [ ]:
# --- Standardise, fingerprint, deduplicate ---------------------------------------------
_norm = rdMolStandardize.Normalizer()
_frag = rdMolStandardize.LargestFragmentChooser()
_unch = rdMolStandardize.Uncharger()


def standardise(mol):
    """Normalise -> largest fragment -> uncharge. Returns None if anything fails."""
    if mol is None:
        return None
    try:
        m = _norm.normalize(mol)
        m = _frag.choose(m)
        m = _unch.uncharge(m)
        Chem.SanitizeMol(m)
        return m
    except Exception:
        return None


def morgan_fp(mol):
    return AllChem.GetMorganFingerprintAsBitVect(
        mol, CONFIG.morgan_radius, nBits=CONFIG.morgan_n_bits)


def fp_to_row(fp) -> np.ndarray:
    arr = np.zeros((CONFIG.morgan_n_bits,), dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr


def inchikey(mol) -> Optional[str]:
    try:
        return Chem.inchi.MolToInchiKey(mol)
    except Exception:
        return None


def featurize_smiles(smiles_list, desc="Featurizing"):
    """-> (keep_mask, fps, inchikeys). Nothing is dropped here; the caller decides."""
    keep, fps, keys = [], [], []
    for smi in tqdm(list(smiles_list), desc=desc, unit="mol"):
        m = standardise(Chem.MolFromSmiles(smi) if isinstance(smi, str) else None)
        if m is None:
            keep.append(False)
            continue
        try:
            fps.append(morgan_fp(m))
            keys.append(inchikey(m) or Chem.MolToSmiles(m))
            keep.append(True)
        except Exception:
            keep.append(False)
    return np.asarray(keep, dtype=bool), fps, keys


_keep, train_fps, train_keys = featurize_smiles(train_df["smiles"],
                                                desc="Training set")
n0 = len(train_df)
train_df = train_df[_keep].reset_index(drop=True)
train_df["_key"] = train_keys
if n0 != len(train_df):
    print(f"  {n0 - len(train_df)} training structures failed to parse and were dropped")

if CONFIG.dedupe_on_inchikey:
    _first = ~train_df["_key"].duplicated()
    if (~_first).any():
        print(f"  deduplicated on standardised InChIKey: {len(train_df):,} -> "
              f"{int(_first.sum()):,}")
        train_fps = [f for f, k in zip(train_fps, _first) if k]
        train_df = train_df[_first].reset_index(drop=True)

X_train = np.stack([fp_to_row(f) for f in train_fps]).astype(np.float32)
y_train = train_df["y"].values.astype(int)
TRAIN_KEYS = set(train_df["_key"])

print(f"\nDesign matrix: {X_train.shape} (uint8 bits as float32)")
print(f"  active {int((y_train == 1).sum()):,} | inactive {int((y_train == 0).sum()):,} "
      f"| base rate {y_train.mean():.3f}")
print(f"  mean bits set per molecule: {X_train.sum(axis=1).mean():.1f} of "
      f"{CONFIG.morgan_n_bits}")
_dead = int((X_train.sum(axis=0) == 0).sum())
print(f"  bits never set in the training set: {_dead:,} of {CONFIG.morgan_n_bits:,} "
      f"({100 * _dead / CONFIG.morgan_n_bits:.0f}%)")
print("  Those dead bits are the mechanism behind the domain problem: any library")
print("  compound whose structure lives in them is invisible to the model, which can")
print("  only score what its training chemistry taught it to see.")

## 4. Part C — The two models

LightGBM and RandomForest, both with balanced class weights. The grids are small on purpose:
these two were already selected on F9's evidence, and the point here is to confirm the
hyperparameters still hold under `fp_only` rather than to search again.

Tuning uses stratified 10-fold CV on the labelled set and is ranked by the AutoQSAR
gap-aware score, `(acc_test − 0.5)·(1 − |acc_train − acc_test|) + 0.5`, after Dixon et al.
(2016). It is worth knowing that this score systematically prefers under-fit models — it
penalises the train/test gap even when the held-out score is better — so the cell prints how
much held-out AUC the gap penalty cost whenever that exceeds 0.005. These CV numbers come
from a random split of a congeneric dataset and carry the usual analogue leakage; **Part D
is the honest estimate**, and it came out 0.06–0.09 lower in F9.

Set `CONFIG.retune = False` to skip this and use F9's parameters directly.

In [ ]:
# --- The two models --------------------------------------------------------------------
F9_PARAMS = {
    "LightGBM": {"n_estimators": 200, "learning_rate": 0.05, "num_leaves": 31,
                 "max_depth": -1},
    "RandomForest": {"n_estimators": 300, "max_features": 0.05, "min_samples_leaf": 3},
}

GRIDS = {
    "LightGBM": [{"n_estimators": n, "learning_rate": lr, "num_leaves": nl,
                  "max_depth": -1}
                 for n in (100, 200, 400) for lr in (0.02, 0.05, 0.1)
                 for nl in (15, 31, 63)],
    "RandomForest": [{"n_estimators": n, "max_features": mf, "min_samples_leaf": ml}
                     for n in (300, 600) for mf in ("sqrt", 0.05, 0.1)
                     for ml in (1, 3)],
}


def make_model(name: str, params: dict):
    cw = "balanced" if CONFIG.class_weight_balanced else None
    if name == "LightGBM":
        return LGBMClassifier(class_weight=cw, random_state=CONFIG.random_state,
                              verbose=-1, n_jobs=-1, **params)
    if name == "RandomForest":
        return RandomForestClassifier(class_weight=cw, random_state=CONFIG.random_state,
                                      n_jobs=-1, **params)
    raise ValueError(name)


def proba(model, X) -> np.ndarray:
    """P(active), robust to a fold that saw only one class."""
    p = model.predict_proba(X)
    classes = list(getattr(model, "classes_", [0, 1]))
    if POSITIVE_CLASS in classes:
        return p[:, classes.index(POSITIVE_CLASS)]
    return np.full(len(X), float(classes[0] == POSITIVE_CLASS))


def autoqsar_score(acc_train: float, acc_test: float) -> float:
    return (acc_test - 0.5) * (1 - abs(acc_train - acc_test)) + 0.5


def tune(name: str) -> dict:
    """Stratified CV over a small grid, ranked by the gap-aware AutoQSAR score."""
    skf = StratifiedKFold(n_splits=CONFIG.tuning_cv_folds, shuffle=True,
                          random_state=CONFIG.random_state)
    rows = []
    for params in tqdm(GRIDS[name], desc=f"  {name}", unit="cfg", leave=False):
        tr_auc, te_auc = [], []
        for tr, te in skf.split(X_train, y_train):
            m = make_model(name, params).fit(X_train[tr], y_train[tr])
            tr_auc.append(roc_auc_score(y_train[tr], proba(m, X_train[tr])))
            te_auc.append(roc_auc_score(y_train[te], proba(m, X_train[te])))
        a_tr, a_te = float(np.mean(tr_auc)), float(np.mean(te_auc))
        rows.append({"params": params, "train_auc": a_tr, "test_auc": a_te,
                     "autoqsar": autoqsar_score(a_tr, a_te)})
    df = pd.DataFrame(rows).sort_values("autoqsar", ascending=False)
    best = df.iloc[0]
    best_by_test = df.sort_values("test_auc", ascending=False).iloc[0]
    print(f"  {name:<14s} AutoQSAR={best['autoqsar']:.3f} "
          f"(train {best['train_auc']:.3f}, test {best['test_auc']:.3f})  {best['params']}")
    cost = float(best_by_test["test_auc"] - best["test_auc"])
    if cost > 0.005:
        print(f"    [gap penalty cost {cost:+.3f} test AUC — it chose the flatter model; "
              f"{best_by_test['params']} scored {best_by_test['test_auc']:.3f}]")
    return dict(best["params"])


BEST_PARAMS: Dict[str, dict] = {}
if CONFIG.retune:
    print(f"=== Tuning on {len(train_df)} labelled compounds, "
          f"{CONFIG.tuning_cv_folds}-fold stratified CV ===")
    for name in tqdm(CONFIG.models, desc="Tuning", unit="model"):
        BEST_PARAMS[name] = tune(name)
else:
    BEST_PARAMS = {n: dict(F9_PARAMS[n]) for n in CONFIG.models}
    print("retune = False — using F9's hyperparameters:")
    for n, p in BEST_PARAMS.items():
        print(f"  {n:<14s} {p}")

print("\nThese CV numbers come from random splits of a congeneric set and are optimistic.")
print("Part D's leave-one-cluster-out estimate is the one to quote.")

## 5. Part D — Honest evaluation, and where the active/inactive line goes

Three steps, and the third is the one this notebook exists for.

**Butina clustering** at Tanimoto distance 0.4 groups the training set into chemical
families. Clusters large and balanced enough to score get their own fold; the rest are
pooled into size-balanced "small families" folds. Holding out a whole family at a time is
what makes the estimate prospective: the model is asked about chemistry it has not seen a
relative of, which is the situation a library screen actually puts it in. A random split
leaves close analogues on both sides and in F9 inflated ROC-AUC by 0.06–0.13 depending on
the model.

**Out-of-fold predictions** are then concatenated, so every labelled compound is scored
exactly once by a model that never saw its family, and the pooled metrics are computed on
that single column. Pooled is what to quote; the per-fold spread says which families are
reachable, not how well the model works.

**The threshold** is chosen on those out-of-fold predictions. The sweep prints precision,
recall, MCC and F1 across the whole range, together with how many of the 290,000 library
compounds each cut would call active, so the choice can be made against whatever you
actually intend to do with the list — a threshold that maximises MCC is the right default,
but a threshold set for precision ≥ 0.9 is the right choice if the next step is expensive.

Calibration is a Platt fit on the same out-of-fold column. One caveat carried over from F9:
the out-of-fold column is assembled from one calibration map *per fold*, so pooling can
reorder compounds that sat in different folds, and the calibrated pooled AUC can therefore
shift. Quote the **raw** AUC for discrimination and the calibrated numbers for probability
quality. The library is scored with a single deployment calibrator, so its ranking is
untouched.

In [ ]:
# --- Metrics ---------------------------------------------------------------------------
def _auc_safe(y_true, p) -> float:
    y_true = np.asarray(y_true)
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, p))


def enrichment_factor(y_true, scores, frac: float) -> float:
    y_true = np.asarray(y_true)
    base = (y_true == POSITIVE_CLASS).mean()
    if base == 0 or len(y_true) == 0:
        return float("nan")
    k = max(1, int(round(frac * len(y_true))))
    top = np.argsort(-np.asarray(scores))[:k]
    return float((y_true[top] == POSITIVE_CLASS).mean() / base)


def precision_at_n(y_true, scores, n: int) -> float:
    y_true = np.asarray(y_true)
    n = min(n, len(y_true))
    if n == 0:
        return float("nan")
    top = np.argsort(-np.asarray(scores))[:n]
    return float((y_true[top] == POSITIVE_CLASS).mean())


def expected_calibration_error(y_bin, p, n_bins=10) -> float:
    edges = np.linspace(0, 1, n_bins + 1)
    ece, n = 0.0, len(p)
    if n == 0:
        return float("nan")
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (p >= lo) & (p < hi) if hi < 1 else (p >= lo) & (p <= hi)
        if m.sum() == 0:
            continue
        ece += (m.sum() / n) * abs(y_bin[m].mean() - p[m].mean())
    return float(ece)


def classification_metrics(y_true, p, threshold: float = 0.5) -> Dict[str, float]:
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p, dtype=float)
    y_pred = (p >= threshold).astype(int)
    multi = len(np.unique(y_true)) > 1
    out = {
        "n": int(len(y_true)),
        "n_active": int((y_true == POSITIVE_CLASS).sum()),
        "base_rate": float((y_true == POSITIVE_CLASS).mean()) if len(y_true) else np.nan,
        "roc_auc": _auc_safe(y_true, p),
        "pr_auc": float(average_precision_score(y_true, p)) if multi else np.nan,
        "balanced_acc": float(balanced_accuracy_score(y_true, y_pred)) if multi else np.nan,
        "mcc": float(matthews_corrcoef(y_true, y_pred)) if multi else np.nan,
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "brier": float(brier_score_loss(y_true, np.clip(p, 0, 1))) if len(y_true) else np.nan,
        "ECE": expected_calibration_error(y_true, np.clip(p, 0, 1)) if len(y_true) else np.nan,
    }
    for frac in CONFIG.enrichment_fractions:
        out[f"EF@{int(frac * 100)}%"] = enrichment_factor(y_true, p, frac)
    for n in CONFIG.precision_at_n:
        out[f"precision@{n}"] = precision_at_n(y_true, p, n)
    return out


print("Metrics defined.")

In [ ]:
# --- Butina folds ----------------------------------------------------------------------
from rdkit.ML.Cluster import Butina


def butina_cluster_groups(fps, cutoff: float = 0.4) -> np.ndarray:
    n = len(fps)
    dists: List[float] = []
    for i in tqdm(range(1, n), desc=f"Butina distances (cutoff={cutoff})", leave=False):
        sims = DataStructs.BulkTanimotoSimilarity(fps[i], fps[:i])
        dists.extend(1.0 - s for s in sims)
    clusters = Butina.ClusterData(dists, n, cutoff, isDistData=True)
    groups = np.empty(n, dtype=int)
    for ci, members in enumerate(clusters):
        for idx in members:
            groups[idx] = ci
    return groups


def mean_min_dissimilarity(idx_in, idx_out, desc="distance") -> float:
    """Mean over `idx_in` of 1 - max Tanimoto to anything in `idx_out`."""
    if not len(idx_in) or not len(idx_out):
        return float("nan")
    ref = [train_fps[i] for i in idx_out]
    vals = []
    for i in tqdm(idx_in, desc=desc, leave=False, unit="mol"):
        vals.append(1.0 - max(DataStructs.BulkTanimotoSimilarity(train_fps[i], ref)))
    return float(np.mean(vals))


butina_groups = butina_cluster_groups(train_fps, cutoff=CONFIG.butina_cutoff)
_sizes = pd.Series(butina_groups).value_counts()
print(f"Butina clustering of {len(train_df):,} compounds (cutoff {CONFIG.butina_cutoff}):")
print(f"  {len(_sizes)} clusters | largest {_sizes.iloc[0]} "
      f"({100 * _sizes.iloc[0] / len(butina_groups):.1f}%) | "
      f"median {int(_sizes.median())} | singletons {int((_sizes == 1).sum())}")


def _scorable(mask) -> bool:
    lab = y_train[mask]
    return ((lab == 1).sum() >= CONFIG.loco_min_fold_actives
            and (lab == 0).sum() >= CONFIG.loco_min_fold_actives)


own_fold, pooled_clusters, rejected = [], [], []
for c in _sizes.index:
    c = int(c)
    mask = butina_groups == c
    if _sizes[c] < CONFIG.loco_min_cluster_size:
        pooled_clusters.append(c)
    elif not _scorable(mask):
        pooled_clusters.append(c)
        rejected.append((c, int(_sizes[c])))
    elif len(own_fold) < CONFIG.loco_max_clusters:
        own_fold.append(c)
    else:
        pooled_clusters.append(c)

print(f"\n  {len(own_fold)} clusters get their own fold (>= "
      f"{CONFIG.loco_min_cluster_size} compounds, >= "
      f"{CONFIG.loco_min_fold_actives} of each class)")
if rejected:
    print(f"  {len(rejected)} large-enough clusters pooled for being too unbalanced to "
          f"score: " + ", ".join(f"cluster {c} (n={n})" for c, n in rejected[:6]))
print(f"  {len(pooled_clusters)} clusters pooled into "
      f"{CONFIG.loco_n_remainder_folds} size-balanced 'small families' folds")

_bins = [[] for _ in range(max(1, CONFIG.loco_n_remainder_folds))]
_bin_sizes = [0] * len(_bins)
for c in sorted(pooled_clusters, key=lambda c: -_sizes[c]):
    j = int(np.argmin(_bin_sizes))
    _bins[j].append(c)
    _bin_sizes[j] += int(_sizes[c])

LOCO_FOLDS = []
for c in own_fold:
    LOCO_FOLDS.append({"fold": f"cluster {c}", "mask": butina_groups == c})
for j, cs in enumerate(_bins):
    if cs:
        LOCO_FOLDS.append({"fold": f"small families #{j + 1}",
                           "mask": np.isin(butina_groups, cs)})

for f in tqdm(LOCO_FOLDS, desc="Fold metadata", unit="fold"):
    m = f["mask"]
    f["n"] = int(m.sum())
    f["n_active"] = int((y_train[m] == 1).sum())
    f["base_rate"] = round(float(y_train[m].mean()), 3)
    srcs = train_df[m]["source_tag"].value_counts()
    f["dominant_source"] = srcs.index[0] if len(srcs) else "?"
    f["source_share"] = round(float(srcs.iloc[0] / int(m.sum())), 2) if len(srcs) else np.nan
    f["mean_tanimoto_to_train"] = round(
        mean_min_dissimilarity(np.flatnonzero(m), np.flatnonzero(~m),
                               desc=f"  {f['fold']}"), 3)

loco_folds_df = pd.DataFrame([{k: v for k, v in f.items() if k != "mask"}
                              for f in LOCO_FOLDS]).set_index("fold")
print(f"\n{len(LOCO_FOLDS)} folds covering "
      f"{int(sum(f['n'] for f in LOCO_FOLDS)):,} of {len(train_df):,} compounds:")
display(loco_folds_df)

_single = loco_folds_df[(loco_folds_df["source_share"] >= 0.6)
                        & (~loco_folds_df["dominant_source"].isin(GENERIC_SOURCE_TAGS))]
if len(_single):
    print("Folds dominated by a single filing — holding one out removes a large,")
    print("chemically narrow block of training data at the same time as it poses an")
    print("unfamiliar test set, so read these rows as the hardest case, not the typical one:")
    for name, r in _single.iterrows():
        print(f"  {name}: {r['n']} compounds, {100 * r['source_share']:.0f}% from "
              f"{r['dominant_source']}, mean Tanimoto distance "
              f"{r['mean_tanimoto_to_train']}")

In [ ]:
# --- Run the folds, collect out-of-fold predictions ------------------------------------
oof = {name: np.full(len(train_df), np.nan) for name in CONFIG.models}
per_fold = []

for f in tqdm(LOCO_FOLDS, desc="Leave-one-cluster-out", unit="fold"):
    te = f["mask"]
    tr = ~te
    if len(np.unique(y_train[tr])) < 2:
        print(f"  {f['fold']}: training side is single-class, skipped")
        continue
    for name in CONFIG.models:
        m = make_model(name, BEST_PARAMS[name]).fit(X_train[tr], y_train[tr])
        p = proba(m, X_train[te])
        oof[name][te] = p
        row = classification_metrics(y_train[te], p)
        row.update({"fold": f["fold"], "model": name,
                    "mean_tanimoto_to_train": f["mean_tanimoto_to_train"]})
        per_fold.append(row)

per_fold_df = (pd.DataFrame(per_fold)
               .set_index(["fold", "model"])
               .sort_index())
_cols = ["n", "n_active", "base_rate", "mean_tanimoto_to_train", "roc_auc", "pr_auc",
         "balanced_acc", "mcc", "brier", "ECE"]
print("=" * 110)
print("PER-FOLD — which chemical families are reachable")
print("=" * 110)
display(per_fold_df[_cols].round(3))

covered = np.isfinite(oof[CONFIG.models[0]])
print(f"\nOut-of-fold predictions for {int(covered.sum()):,} of {len(train_df):,} "
      f"compounds.")

In [ ]:
# --- Pooled results, and the leakage reference -----------------------------------------
print("=" * 110)
print("POOLED — every compound scored once, by a model blind to its whole family.")
print("QUOTE THESE.")
print("=" * 110)
pooled = []
for name in CONFIG.models:
    p = oof[name][covered]
    row = classification_metrics(y_train[covered], p)
    row["model"] = name
    pooled.append(row)
pooled_df = pd.DataFrame(pooled).set_index("model")
display(pooled_df.round(3))

# random-split reference, to size the optimism
print("\n" + "=" * 110)
print("RANDOM-SPLIT REFERENCE — the gap to pooled is the analogue leakage")
print("=" * 110)
_ref = {name: [] for name in CONFIG.models}
_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=CONFIG.random_state)
for tr, te in tqdm(list(_skf.split(X_train, y_train)), desc="Random-split reference",
                   unit="fold"):
    for name in CONFIG.models:
        m = make_model(name, BEST_PARAMS[name]).fit(X_train[tr], y_train[tr])
        _ref[name].append(_auc_safe(y_train[te], proba(m, X_train[te])))
print()
for name in CONFIG.models:
    r = float(np.mean(_ref[name]))
    pl = float(pooled_df.loc[name, "roc_auc"])
    print(f"  {name:<14s} random {r:.3f}  ->  pooled {pl:.3f}   [{pl - r:+.3f}]")
print("\nThe pooled column is the one that reflects screening an unfamiliar library.")

### 5b. The threshold sweep

Everything below is computed on the out-of-fold predictions, so each number is an estimate
of what that cut would do on chemistry the model has not seen a relative of.

`n_called_est` extrapolates the called fraction to the full library. Treat it as the
order of magnitude rather than a forecast: the library's score distribution is not the
training set's, precisely because its chemistry is different, and Part F reports the actual
counts once the library has been scored.

In [ ]:
# --- Choose the decision threshold on out-of-fold predictions --------------------------
CALIBRATORS: Dict[str, Optional[object]] = {name: None for name in CONFIG.models}
oof_cal = {}
for name in CONFIG.models:
    p = oof[name][covered]
    if CONFIG.calibrate:
        cal = LogisticRegression(C=1e6, solver="lbfgs")
        cal.fit(p.reshape(-1, 1), y_train[covered])
        CALIBRATORS[name] = cal
        oof_cal[name] = cal.predict_proba(p.reshape(-1, 1))[:, 1]
        print(f"{name:<14s} Platt calibrator: p_cal = sigmoid("
              f"{float(cal.coef_[0][0]):+.3f} * p {float(cal.intercept_[0]):+.3f})"
              f"   ECE {expected_calibration_error(y_train[covered], p):.3f}"
              f" -> {expected_calibration_error(y_train[covered], oof_cal[name]):.3f}")
        if float(cal.coef_[0][0]) < 0:
            print("  !! NEGATIVE SLOPE — this calibrator inverts the ranking. Something is")
            print("     wrong upstream; do not deploy it.")
    else:
        oof_cal[name] = p

GRID = np.round(np.arange(0.05, 0.96, 0.01), 2)
sweeps = {}
for name in CONFIG.models:
    p, yy = oof_cal[name], y_train[covered]
    rows = []
    for t in GRID:
        yp = (p >= t).astype(int)
        rows.append({
            "threshold": t,
            "precision": float(precision_score(yy, yp, zero_division=0)),
            "recall": float(recall_score(yy, yp, zero_division=0)),
            "f1": float(f1_score(yy, yp, zero_division=0)),
            "mcc": float(matthews_corrcoef(yy, yp)) if len(np.unique(yp)) > 1 else 0.0,
            "balanced_acc": float(balanced_accuracy_score(yy, yp)),
            "called_frac": float(yp.mean()),
        })
    sweeps[name] = pd.DataFrame(rows)

OBJ = CONFIG.threshold_objective
THRESHOLDS: Dict[str, float] = {}
print("\n" + "=" * 110)
print(f"THRESHOLD SWEEP — chosen by max {OBJ} on out-of-fold predictions")
print("=" * 110)
CAP = float(CONFIG.threshold_max_called_frac)
for name in CONFIG.models:
    s = sweeps[name]
    unconstrained = s.loc[s[OBJ].idxmax()]
    # Restrict to cuts that actually produce a shortlist. An unconstrained argmax of MCC
    # or F1 on a weak model lands on a permissive cut that calls nearly everything.
    ok = s[s["called_frac"] <= CAP]
    capped = len(ok) > 0
    best = ok.loc[ok[OBJ].idxmax()] if capped else s.iloc[-1]
    THRESHOLDS[name] = (float(CONFIG.threshold_override)
                        if CONFIG.threshold_override is not None
                        else float(best["threshold"]))
    print(f"\n{name}")
    print(f"  max {OBJ:<12s} at p >= {best['threshold']:.2f}  "
          f"precision {best['precision']:.3f}  recall {best['recall']:.3f}  "
          f"MCC {best['mcc']:.3f}  calls {100 * best['called_frac']:.1f}%")
    if not capped:
        print(f"  !! NO threshold calls <= {100 * CAP:.0f}% of compounds active. Using the")
        print(f"     most stringent cut in the sweep (p >= {best['threshold']:.2f}), which")
        print(f"     still calls {100 * best['called_frac']:.1f}%. The model is not")
        print("     separating the classes; treat the output as a weak ranking only.")
    elif float(unconstrained["threshold"]) != float(best["threshold"]):
        print(f"  (unconstrained max {OBJ} was at p >= {unconstrained['threshold']:.2f}, "
              f"but it calls {100 * unconstrained['called_frac']:.1f}% active —")
        print(f"   above the {100 * CAP:.0f}% cap — so it was rejected. That cut maximises")
        print(f"   {OBJ} by being permissive, which is not a shortlist.)")
    if best["recall"] >= 0.99:
        print("  !! recall is ~1.0 at this cut: it is calling essentially every active")
        print("     AND most inactives. Lower threshold_max_called_frac.")
    for target in (0.80, 0.90, 0.95):
        ok = s[s["precision"] >= target]
        if len(ok):
            r = ok.loc[ok["recall"].idxmax()]
            print(f"  precision >= {target:.2f} at p >= {r['threshold']:.2f}  "
                  f"(recall {r['recall']:.3f}, calls {100 * r['called_frac']:.1f}% "
                  f"~ {int(round(r['called_frac'] * 290000)):,} of 290,000)")
        else:
            print(f"  precision >= {target:.2f} is NOT reachable at any threshold "
                  f"(max {s['precision'].max():.3f})")
    if CONFIG.threshold_override is not None:
        print(f"  -> threshold_override is set: using p >= {THRESHOLDS[name]:.2f}")
    print(f"  USING p >= {THRESHOLDS[name]:.2f}")

_show = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
for name in CONFIG.models:
    s = sweeps[name]
    print(f"\n{name} — sweep at selected cuts:")
    display(s[s["threshold"].isin(_show)].set_index("threshold").round(3))

fig, axes = plt.subplots(1, len(CONFIG.models), figsize=(6.2 * len(CONFIG.models), 4.2),
                         squeeze=False)
for ax, name in zip(axes[0], CONFIG.models):
    s = sweeps[name]
    for col, style in [("precision", "-"), ("recall", "--"), ("mcc", "-.")]:
        ax.plot(s["threshold"], s[col], style, label=col)
    ax.axvline(THRESHOLDS[name], color="k", lw=1, ls=":",
               label=f"chosen {THRESHOLDS[name]:.2f}")
    ax.set_title(f"{name} — out-of-fold")
    ax.set_xlabel("P(active) threshold")
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print("\nMetrics at the chosen thresholds (out-of-fold, calibrated):")
_at = []
for name in CONFIG.models:
    row = classification_metrics(y_train[covered], oof_cal[name], THRESHOLDS[name])
    row["model"] = name
    row["threshold"] = THRESHOLDS[name]
    _at.append(row)
display(pd.DataFrame(_at).set_index("model")[
    ["threshold", "roc_auc", "pr_auc", "precision", "recall", "f1", "mcc",
     "balanced_acc", "ECE"]].round(3))

In [ ]:
# --- Refit on everything, for deployment -----------------------------------------------
# Evaluation and the threshold came from out-of-fold predictions; the deployed model is
# then refitted on all the labelled data, which is the standard arrangement and is why
# the two must not be swapped.
FINAL_MODELS = {}
for name in tqdm(CONFIG.models, desc="Final fit", unit="model"):
    FINAL_MODELS[name] = make_model(name, BEST_PARAMS[name]).fit(X_train, y_train)

BOOT_MODELS = {name: [] for name in CONFIG.models}
if CONFIG.n_bootstrap_shortlist:
    rng = np.random.RandomState(CONFIG.random_state)
    for name in CONFIG.models:
        for b in tqdm(range(CONFIG.n_bootstrap_shortlist),
                      desc=f"Bootstrap {name}", unit="model", leave=False):
            idx = rng.choice(len(y_train), len(y_train), replace=True)
            if len(np.unique(y_train[idx])) < 2:
                continue
            BOOT_MODELS[name].append(
                make_model(name, BEST_PARAMS[name]).fit(X_train[idx], y_train[idx]))
    print(f"Bootstrap ensembles: "
          + ", ".join(f"{n} x{len(BOOT_MODELS[n])}" for n in CONFIG.models))
    print("These are used for a spread on the shortlist only — running 40 models over")
    print("290,000 compounds would cost far more than the uncertainty estimate is worth.")

print("\nDeployment configuration:")
for name in CONFIG.models:
    print(f"  {name:<14s} params {BEST_PARAMS[name]}")
    print(f"  {'':<14s} threshold p >= {THRESHOLDS[name]:.2f}, "
          f"calibrated: {CALIBRATORS[name] is not None}")

## 6. Part E — Score the full library

The SDF is streamed with `ForwardSDMolSupplier` and processed in chunks of
`screening_chunk_size`, so at no point is the full 290,000 × 2048 feature matrix in memory —
that alone would be 2.4 GB as float32, before anything else. Each chunk is standardised,
fingerprinted, scored by both models, and appended to a CSV on disk; only a small
per-compound summary is kept in memory for Part F.

Deduplication happens on the way through, against a running set of standardised InChIKeys.
Compounds whose standardised structure is already in the **training set** are flagged
`in_training` rather than dropped — they are a free positive control, and where they rank is
the first thing to look at in Part F.

Set `CONFIG.screening_limit` to a few thousand for a trial run before committing to the
whole file.

In [ ]:
# --- Stream the library and score it ----------------------------------------------------
def open_sdf(path_str: str):
    """Returns (supplier, resolved_path). Accepts .sdf and .sdf.gz; a leading '/' that
    does not resolve is retried as a relative path, which is how the Colab versions of
    these notebooks referred to a file at the Drive root."""
    p = Path(path_str)
    if not p.exists() and path_str.startswith(("/", "\\")):
        alt = Path("." + path_str)
        if alt.exists():
            print(f"  {path_str} not found; using {alt}")
            p = alt
    if not p.exists():
        raise FileNotFoundError(
            f"Screening library not found at {path_str}. Set "
            f"CONFIG.screening_library_path to the real location — this notebook "
            f"deliberately does not substitute a demo file, because a silent fallback "
            f"would produce a plausible-looking ranking of the wrong molecules.")
    fh = gzip.open(p, "rb") if p.name.lower().endswith(".gz") else open(p, "rb")
    return Chem.ForwardSDMolSupplier(fh), p


def mol_identifier(mol, i: int, stem: str) -> str:
    if mol.HasProp("_Name") and mol.GetProp("_Name").strip():
        return mol.GetProp("_Name").strip()
    for key in ("idnumber", "ID", "id", "Catalog ID", "catalog_id", "ZINC_ID", "Name"):
        if mol.HasProp(key) and mol.GetProp(key).strip():
            return mol.GetProp(key).strip()
    return f"{stem}:{i + 1}"


OUT_DIR = Path(CONFIG.output_dir)
PRED_CSV = OUT_DIR / "library_predictions.csv"
if PRED_CSV.exists():
    PRED_CSV.unlink()

suppl, lib_path = open_sdf(CONFIG.screening_library_path)
print(f"Streaming {lib_path} in chunks of {CONFIG.screening_chunk_size:,}.")
print("Fingerprinting is the slow step; the models themselves are fast.\n")

seen_keys = set()
n_read = n_fail = n_dup = n_written = 0
buf_ids, buf_smi, buf_fps, buf_keys = [], [], [], []
first_write = True
_stem = lib_path.name


def flush():
    """Score one buffered chunk and append it to the CSV."""
    global first_write, n_written
    if not buf_fps:
        return
    X = np.stack([fp_to_row(f) for f in buf_fps]).astype(np.float32)
    out = pd.DataFrame({"compound_id": buf_ids, "smiles": buf_smi,
                        "inchikey": buf_keys})
    for name in CONFIG.models:
        p = proba(FINAL_MODELS[name], X)
        cal = CALIBRATORS[name]
        if cal is not None:
            p = cal.predict_proba(p.reshape(-1, 1))[:, 1]
        col = name.replace(" ", "")
        out[f"p_{col}"] = p.round(5)
        out[f"call_{col}"] = (p >= THRESHOLDS[name]).astype(int)
    out["n_models_active"] = sum(out[f"call_{n.replace(' ', '')}"]
                                 for n in CONFIG.models)
    out["p_mean"] = out[[f"p_{n.replace(' ', '')}" for n in CONFIG.models]].mean(
        axis=1).round(5)
    out["in_training"] = [k in TRAIN_KEYS for k in buf_keys]
    out.to_csv(PRED_CSV, mode="a", header=first_write, index=False)
    first_write = False
    n_written += len(out)
    buf_ids.clear(); buf_smi.clear(); buf_fps.clear(); buf_keys.clear()


bar = tqdm(desc="Screening", unit="mol")
for i, mol in enumerate(suppl):
    if CONFIG.screening_limit and n_read >= CONFIG.screening_limit:
        break
    n_read += 1
    bar.update(1)
    std = standardise(mol)
    if std is None:
        n_fail += 1
        continue
    key = inchikey(std) or Chem.MolToSmiles(std)
    if CONFIG.dedupe_on_inchikey:
        if key in seen_keys:
            n_dup += 1
            continue
        seen_keys.add(key)
    try:
        fp = morgan_fp(std)
    except Exception:
        n_fail += 1
        continue
    buf_ids.append(mol_identifier(mol, i, _stem))
    buf_smi.append(Chem.MolToSmiles(std))
    buf_fps.append(fp)
    buf_keys.append(key)
    if len(buf_fps) >= CONFIG.screening_chunk_size:
        flush()
        bar.set_postfix(scored=f"{n_written:,}")
flush()
bar.close()

print(f"\n  records read            : {n_read:,}")
print(f"  unparseable / failed    : {n_fail:,}")
print(f"  duplicate structures    : {n_dup:,} "
      f"({100 * n_dup / max(n_read, 1):.1f}% of the file)")
print(f"  unique compounds scored : {n_written:,}")
print(f"  written to              : {PRED_CSV}")

## 7. Part F — The calls

Four things, in order of how much they should change your mind.

1. **The positive control.** Any library compound whose standardised structure matches a
   training compound is flagged `in_training`. Those are known answers that arrived through
   the same pipeline as everything else, and if they do not rank near the top, nothing
   further down is worth reading. Catalogues of this size usually contain a handful.
2. **How many got called, and the agreement between the two models.** `n_models_active` is
   0, 1 or 2. Compounds both models call active are the defensible list; compounds only one
   calls are where the two inductive biases disagree, and the disagreement rate is a
   measure of how far outside its training distribution the library sits.
3. **The score distribution.** A screen of unfamiliar chemistry should produce a
   distribution piled up near zero. A library that comes out looking as active as the
   training set is a warning sign, not a success.
4. **The shortlist**, with a bootstrap spread.

In [ ]:
# --- Read the predictions back and summarise -------------------------------------------
res = pd.read_csv(PRED_CSV)
MCOLS = [f"p_{n.replace(' ', '')}" for n in CONFIG.models]
CCOLS = [f"call_{n.replace(' ', '')}" for n in CONFIG.models]
print(f"{len(res):,} compounds scored.\n")

# ---------------------------------------------------------------- 1. positive control
print("=" * 100)
print("1. POSITIVE CONTROL — library compounds that are also in the training set")
print("=" * 100)
known = res[res["in_training"]]
if len(known) == 0:
    print("  None. This catalogue shares no standardised structure with the training set,")
    print("  so there is no internal check on the ranking. That is consistent with")
    print("  everything F6-F9 found about the overlap, but it does mean the numbers below")
    print("  rest entirely on the out-of-fold estimate in Part D.")
else:
    res["_rank_mean"] = res["p_mean"].rank(ascending=False)
    k = res[res["in_training"]].copy()
    k["percentile"] = (100 * (1 - k["_rank_mean"] / len(res))).round(2)
    print(f"  {len(k)} compound(s) in both. Where they rank:")
    display(k[["compound_id", "p_mean"] + CCOLS + ["percentile"]]
            .sort_values("p_mean", ascending=False).head(20))
    med = float(k["percentile"].median())
    print(f"  median percentile: {med:.1f}")
    if med < 90:
        print("  !! The known compounds do NOT rank near the top. Treat the rest of this")
        print("     ranking as unsupported until that is understood.")
    else:
        print("  The known compounds rank near the top, which is the minimum the ranking")
        print("  has to achieve to be worth reading.")

# ---------------------------------------------------------------- 2. calls and agreement
print("\n" + "=" * 100)
print("2. CALLS, AND WHERE THE TWO MODELS AGREE")
print("=" * 100)
for name in CONFIG.models:
    c = f"call_{name.replace(' ', '')}"
    n = int(res[c].sum())
    print(f"  {name:<14s} threshold p >= {THRESHOLDS[name]:.2f}  ->  "
          f"{n:,} active ({100 * n / len(res):.2f}%)")

tier = res["n_models_active"].value_counts().reindex([2, 1, 0]).fillna(0).astype(int)
print(f"\n  both models active : {tier.get(2, 0):,}   <- the defensible list")
print(f"  one model only     : {tier.get(1, 0):,}   <- the two biases disagree")
print(f"  neither            : {tier.get(0, 0):,}")
if len(CONFIG.models) == 2:
    a, b = (f"call_{n.replace(' ', '')}" for n in CONFIG.models)
    print("\n  Agreement matrix (rows = "
          f"{CONFIG.models[0]}, columns = {CONFIG.models[1]}):")
    display(pd.crosstab(res[a], res[b]))
    both = int(((res[a] == 1) & (res[b] == 1)).sum())
    either = int(((res[a] == 1) | (res[b] == 1)).sum())
    print(f"  Jaccard overlap of the two active sets: "
          f"{both / max(either, 1):.3f}")
    print(f"  Spearman correlation of the two score columns: "
          f"{res[MCOLS[0]].corr(res[MCOLS[1]], method='spearman'):.3f}")
    print("  Two models with different inductive biases agreeing on a compound is a")
    print("  weak but real signal; a low overlap here means most single-model calls are")
    print("  an artefact of one model's bias rather than a property of the molecule.")

# ------------------------------------------------- 2b. what does each list size cost?
print("\n" + "=" * 100)
print("2b. PICK A LIST SIZE — what each one implies")
print("=" * 100)
print("The threshold column is read off the LIBRARY's score distribution; the precision")
print("and recall columns are what that same threshold achieved OUT-OF-FOLD on held-out")
print("training families. Those are the best available estimates, and they are optimistic")
print("here, because the library is further from the training set than any held-out fold.")
_rows = []
for name in CONFIG.models:
    col, s = f"p_{name.replace(' ', '')}", sweeps[name]
    for N in (100, 500, 1000, 5000, 10000, 25000):
        if N > len(res):
            continue
        t = float(res[col].nlargest(N).iloc[-1])
        near = s.iloc[(s["threshold"] - t).abs().idxmin()]
        _rows.append({"model": name, "list_size": N, "implied_threshold": round(t, 3),
                      "oof_precision": round(float(near["precision"]), 3),
                      "oof_recall": round(float(near["recall"]), 3),
                      "oof_mcc": round(float(near["mcc"]), 3)})
if _rows:
    display(pd.DataFrame(_rows).set_index(["model", "list_size"]))
    print("\nWhere oof_precision is 0.000 the implied threshold sits outside the range the")
    print("out-of-fold predictions ever reached — that list size is deeper into the")
    print("library than anything the evaluation can speak to.")

# ---------------------------------------------------------------- 3. score distribution
print("\n" + "=" * 100)
print("3. SCORE DISTRIBUTION")
print("=" * 100)
display(res[MCOLS + ["p_mean"]].describe(
    percentiles=[0.5, 0.9, 0.99, 0.999]).round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for name in CONFIG.models:
    col = f"p_{name.replace(' ', '')}"
    axes[0].hist(res[col], bins=60, histtype="step", lw=1.6, label=f"library — {name}")
for name in CONFIG.models:
    axes[0].hist(oof_cal[name], bins=30, histtype="stepfilled", alpha=0.25,
                 density=False, weights=np.full(len(oof_cal[name]),
                                                len(res) / len(oof_cal[name])),
                 label=f"training (out-of-fold) — {name}")
axes[0].set_yscale("log")
axes[0].set_xlabel("calibrated P(active)")
axes[0].set_ylabel("compounds (log scale)")
axes[0].set_title("Library against the training set")
axes[0].legend(fontsize=7)
axes[1].scatter(res[MCOLS[0]], res[MCOLS[-1]], s=1, alpha=0.08)
axes[1].plot([0, 1], [0, 1], "k--", lw=0.8)
axes[1].set_xlabel(f"P(active) — {CONFIG.models[0]}")
axes[1].set_ylabel(f"P(active) — {CONFIG.models[-1]}")
axes[1].set_title("Do the two models agree per compound?")
plt.tight_layout()
plt.show()

_hi = float((res["p_mean"] >= 0.5).mean())
_tr = float(np.mean([(oof_cal[n] >= 0.5).mean() for n in CONFIG.models]))
print(f"  library at p_mean >= 0.5     : {100 * _hi:.2f}%")
print(f"  training out-of-fold >= 0.5  : {100 * _tr:.2f}%  (base rate "
      f"{y_train.mean():.3f})")
if _hi > _tr:
    print("  !! The library scores HIGHER than the held-out training compounds. That is")
    print("     implausible for chemistry this far from the training set and points at")
    print("     the models keying on something generic rather than the pharmacophore.")
else:
    print("  The library scores well below the held-out training compounds, which is the")
    print("  expected direction for chemistry this far outside the training distribution.")

# ---------------------------------------------------------------- 4. shortlist
print("\n" + "=" * 100)
print(f"4. SHORTLIST — top {CONFIG.shortlist_size:,}")
print("=" * 100)
# Ordering. `p_mean` is reported but NOT used to rank, for two reasons: the two models
# have different thresholds and different probability scales, so averaging the raw
# numbers is the same blending that lost to its best single member on seven consecutive
# endpoints across F4-F9; and it buries the consensus compounds among single-model calls.
# Instead: consensus first, then the mean of the two PERCENTILE ranks, which is scale-free
# and so legitimate to average even when the two probability scales are not comparable.
# That is an ordering rule over two independent opinions, not a combined predictor.
for _name in CONFIG.models:
    res[f"pct_{_name.replace(' ', '')}"] = res[
        f"p_{_name.replace(' ', '')}"].rank(pct=True).round(5)
PCTCOLS = [f"pct_{n.replace(' ', '')}" for n in CONFIG.models]
res["rank_pct_mean"] = res[PCTCOLS].mean(axis=1).round(5)

shortlist = res.sort_values(["n_models_active", "rank_pct_mean"],
                            ascending=[False, False]).head(
    CONFIG.shortlist_size).reset_index(drop=True)
_n_cons = int((shortlist["n_models_active"] == len(CONFIG.models)).sum())
print("  ordered by consensus first, then mean percentile rank across the two models.")
print(f"  {_n_cons} of the top {len(shortlist)} are called active by both models.")

# The consensus set is the direct answer to 'active or inactive', so it is exported
# separately rather than only existing inside the shortlist.
consensus = res[res["n_models_active"] == len(CONFIG.models)].sort_values(
    "rank_pct_mean", ascending=False).reset_index(drop=True)
print(f"  full consensus set (both models active): {len(consensus):,} compounds "
      f"-> consensus_actives.csv")

if CONFIG.n_bootstrap_shortlist and any(BOOT_MODELS.values()):
    _keep, _fps, _ = featurize_smiles(shortlist["smiles"], desc="Shortlist fingerprints")
    Xs = np.stack([fp_to_row(f) for f in _fps]).astype(np.float32)
    shortlist = shortlist[_keep].reset_index(drop=True)
    for name in CONFIG.models:
        if not BOOT_MODELS[name]:
            continue
        P = np.stack([proba(m, Xs) for m in
                      tqdm(BOOT_MODELS[name], desc=f"Bootstrap {name}", leave=False)])
        shortlist[f"spread_{name.replace(' ', '')}"] = P.std(axis=0).round(4)
    print("  Bootstrap spreads measure how much the prediction moves when the training")
    print("  set is resampled. They are model disagreement, not predictive error — a")
    print("  confident wrong answer has a small spread.")

_cols = (["compound_id", "smiles"] + MCOLS + ["p_mean", "rank_pct_mean",
         "n_models_active", "in_training"]
         + [c for c in shortlist.columns if c.startswith("spread_")])
display(shortlist[_cols].head(30))

shortlist[_cols].to_csv(OUT_DIR / "shortlist.csv", index=False)
consensus[[c for c in _cols if c in consensus.columns]].to_csv(
    OUT_DIR / "consensus_actives.csv", index=False)
res["n_models_active"].value_counts().rename("compounds").to_frame().to_csv(
    OUT_DIR / "call_summary.csv")
pooled_df.round(4).to_csv(OUT_DIR / "pooled_loco_metrics.csv")
per_fold_df.round(4).to_csv(OUT_DIR / "per_fold_loco_metrics.csv")
loco_folds_df.to_csv(OUT_DIR / "fold_definitions.csv")
for name in CONFIG.models:
    sweeps[name].round(4).to_csv(
        OUT_DIR / f"threshold_sweep_{name.replace(' ', '_')}.csv", index=False)
train_df.drop(columns=["_key"], errors="ignore").to_csv(
    OUT_DIR / "psma_pki_training_set.csv", index=False)

_safe = lambda s: re.sub(r"[^A-Za-z0-9]+", "_", s).strip("_")
for name in CONFIG.models:
    try:
        joblib.dump(FINAL_MODELS[name], OUT_DIR / f"model_{_safe(name)}.joblib")
    except Exception as e:
        print(f"  (could not pickle {name}: {type(e).__name__}: {e})")

# Plain-data export of everything needed to reproduce a call without unpickling anything.
deploy = {
    "endpoint": PRIMARY,
    "active_threshold_pKi": CONFIG.active_threshold,
    "fingerprint": {"type": "Morgan", "radius": CONFIG.morgan_radius,
                    "n_bits": CONFIG.morgan_n_bits,
                    "standardisation": "Normalizer -> LargestFragmentChooser -> Uncharger"},
    "models": {},
    "training_set": {"n": int(len(train_df)), "n_active": int((y_train == 1).sum()),
                     "base_rate": float(y_train.mean())},
    "evaluation": "leave-one-cluster-out (Butina 0.4), pooled out-of-fold",
}
for name in CONFIG.models:
    cal = CALIBRATORS[name]
    deploy["models"][name] = {
        "params": BEST_PARAMS[name],
        "decision_threshold": THRESHOLDS[name],
        "calibrator": (None if cal is None else
                       {"type": "platt", "coef": float(cal.coef_[0][0]),
                        "intercept": float(cal.intercept_[0]),
                        "apply": "1 / (1 + exp(-(coef * p_raw + intercept)))"}),
        "pooled_loco": {k: (None if pd.isna(v) else float(v)) for k, v in
                        pooled_df.loc[name].items()},
    }
with open(OUT_DIR / "deployment.json", "w") as fh:
    json.dump(deploy, fh, indent=2, default=str)

print(f"\nSaved to {OUT_DIR}:")
for p in sorted(OUT_DIR.iterdir()):
    print(f"  - {p.name}  ({p.stat().st_size:,} bytes)")

## 8. Notes

**What the numbers mean, and what they do not.** The pooled leave-one-cluster-out metrics in
Part D are honest estimates of performance on chemistry the model has not seen a relative
of — within the chemical space the training set spans. The Enamine library largely sits
*outside* that space, which is what F6–F9 established and why the applicability domain is
absent here by instruction rather than by oversight. So the right reading of the output is:
**the ranking is a prioritisation, not a probability of activity.** The calibrated `p_` column
is calibrated against held-out *training* chemistry; nothing in this notebook can calibrate
it against chemistry the training set does not cover.

Concretely, the two claims this pipeline supports are "these compounds are more
PSMA-active-like than those compounds, according to two models that reach 0.78–0.80 ROC-AUC
on unfamiliar families", and "both models independently call these N compounds active at a
threshold that gives roughly 70–80% precision on held-out families". The claim it does not
support is a hit rate for the shortlist.

**If you want to act on this**, the two things that would most change the picture are
cheaper than any modelling change: a counter-screen set (measured inactives from the same
assay, which BindingDB has few of, and which the base rate of 0.51 shows this training set
is missing), and any measured data on a handful of the shortlisted compounds, which would
turn the extrapolation into a calibration point.

**Things deliberately left out**, and where to find them if wanted: the applicability domain
and its four metrics (F6–F9 Part B.2), the Butina leave-one-cluster-out machinery in fuller
form with macro aggregation and a random-split reference per fold (F9 Part E.2), the other
five classifiers and the blend (F9 Part C.2), reaction-based enumeration of a focused
library (F9 Part A.2b), and the atom-level attribution maps (F9 Part E.6).

**Reproducing a single prediction** needs none of this notebook: `deployment.json` carries
the fingerprint definition, both models' hyperparameters, their decision thresholds, and
their Platt coefficients with the formula. The `.joblib` files are a convenience that only
reload where the same library versions are installed.

**Knobs worth turning.** `CONFIG.threshold_objective` switches the cut between MCC, F1 and
balanced accuracy; `CONFIG.threshold_override` forces a specific value, which is what to use
if you pick a cut off the precision targets printed in Part D. `CONFIG.screening_limit` runs
a few thousand compounds first. `CONFIG.retune = False` reuses F9's hyperparameters and
skips the grid. Adding KPLS-DA back means adding one entry to `make_model` and one grid —
it was the best model at the very top of the ranking, and its exclusion here was a close
call made on calibration quality rather than on discrimination.